In [1]:
# =============================
# IMPORTS
# =============================

import os
import re
import csv
import pandas as pd
from PIL import Image
import pytesseract
import pdfplumber
from pdf2image import convert_from_path
from docx import Document

# =============================
# CONFIGURATION
# =============================

POPPLER_PATH = r"C:\Program Files (x86)\poppler-25.12.0\Library\bin"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# =============================
# STUDENT DATA EXTRACTOR CLASS
# =============================

class StudentDataExtractor:

    def __init__(self):
        self.records = []

    # ---------- TEXT EXTRACTION ----------

    def extract_text_from_image(self, image_path):
        try:
            return pytesseract.image_to_string(Image.open(image_path))
        except Exception as e:
            print(f"Image error: {e}")
            return ""

    def extract_text_from_pdf(self, pdf_path):
        text = ""

        # Normal PDF text extraction
        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    if page.extract_text():
                        text += page.extract_text()
        except:
            pass

        # OCR fallback
        if not text.strip():
            try:
                images = convert_from_path(pdf_path, poppler_path=POPPLER_PATH)
                for img in images:
                    text += pytesseract.image_to_string(img)
            except Exception as e:
                print(f"PDF OCR error: {e}")

        return text

    def extract_text_from_docx(self, docx_path):
        try:
            doc = Document(docx_path)
            return "\n".join(p.text for p in doc.paragraphs)
        except Exception as e:
            print(f"DOCX error: {e}")
            return ""

    # ---------- DATA PARSING ----------

    def parse_student_data(self, text):
        data = {}

        patterns = {
            "ID": r"ID\s*[:\-]?\s*(\d+)",
            "Email": r"[\w\.-]+@[\w\.-]+",
            "Phone": r"\b\d{10}\b",
            "DOB": r"\b\d{2}[-/]\d{2}[-/]\d{4}\b",
            "Gender": r"\b(Male|Female|Other)\b",
            "Course": r"(B\.?Tech|BCA|MCA|MBA|BA|CSE|Degree|Inter)",
            "Marks": r"\b\d{2,3}\b"
        }

        name_patterns = [
            r"Name\s*[:\-]?\s*([A-Za-z ]+)",
            r"Student Name\s*[:\-]?\s*([A-Za-z ]+)",
            r"NAME\s*[:\-]?\s*([A-Za-z ]+)"
        ]

        data["Name"] = ""
        for pat in name_patterns:
            m = re.search(pat, text, re.I)
            if m:
                data["Name"] = m.group(1).strip()
                break

        for field, pattern in patterns.items():
            m = re.search(pattern, text, re.I)
            data[field] = m.group().strip() if m else ""

        return data

    # ---------- FILE / DIRECTORY PROCESSING ----------

    def process_path(self, path):
        if os.path.isfile(path):
            self.process_file(path)
        elif os.path.isdir(path):
            for file in os.listdir(path):
                self.process_file(os.path.join(path, file))
        else:
            print("Invalid path")

    def process_file(self, file_path):
        fp = file_path.lower()

        if fp.endswith((".jpg", ".jpeg", ".png")):
            text = self.extract_text_from_image(file_path)
        elif fp.endswith(".pdf"):
            text = self.extract_text_from_pdf(file_path)
        elif fp.endswith(".docx"):
            text = self.extract_text_from_docx(file_path)
        else:
            return

        if text.strip():
            self.records.append(self.parse_student_data(text))
            print(f"✓ Extracted: {os.path.basename(file_path)}")
        else:
            print(f"✗ Failed: {os.path.basename(file_path)}")

    # ---------- DATAFRAME ----------

    def to_dataframe(self):
        if not self.records:
            print("No data available")
            return pd.DataFrame()

        df = pd.DataFrame(self.records)
        return df

    # ---------- SAVE ----------

    def save_to_csv(self, output_file):
        df = self.to_dataframe()

        if df.empty:
            print("No data to save")
            return

        df.to_csv(output_file, index=False, encoding="utf-8")

        print(f"\n✓ CSV Created: {output_file}")
        print(f"✓ Records Saved: {len(df)}")


In [2]:
extractor = StudentDataExtractor()

# Process folder
extractor.process_path(
    r"C:\Users\prane\Downloads\praneeth_project\student_forms"
)

# Convert to DataFrame
df = extractor.to_dataframe()

# Display DataFrame in notebook
df


✓ Extracted: student1.jpg
✓ Extracted: student2.pdf
✓ Extracted: student3.docx


,Name,ID,Email,Phone,DOB,Gender,Course,Marks
0,Kavitha,ID: 1007,kavithareddy@gmail.com,9012345678,03/11/2004,Female,B.Tech,03
1,Venky,ID: 1006,venky@gmail.com,9966064278,01/01/2004,Male,CSE,01
2,Praneeth,ID: 1005,praneethkumar804@gmail.com,7093670268,08/10/2004,Male,BA,08


In [3]:
extractor.save_to_csv(
    r"C:\Users\prane\Downloads\praneeth_project\students_output5.csv"
)



✓ CSV Created: C:\Users\prane\Downloads\praneeth_project\students_output5.csv
✓ Records Saved: 3
